# 1. Data cleaning

Inspect the original loan-default CSV, validate its target and identifier, remove duplicate records, fill missing feature values, and save a clean dataset for the following notebooks. The raw source file is never overwritten.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Loan_default.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "loan_default_cleaned.csv"
TARGET = "Default"
ID_COLUMN = "LoanID"

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]} columns from {RAW_PATH}")
display(df_raw.head())

In [ ]:
print("Column types:")
display(df_raw.dtypes.rename("dtype").to_frame())

quality = pd.DataFrame({
    "Missing values": df_raw.isna().sum(),
    "Unique values": df_raw.nunique(dropna=False),
})
display(quality)
print(f"Exact duplicate rows: {df_raw.duplicated().sum():,}")
if ID_COLUMN in df_raw.columns:
    print(f"Duplicate loan identifiers: {df_raw[ID_COLUMN].duplicated().sum():,}")

## Clean and validate

Duplicate identifiers are reduced to their first record, then exact duplicate rows are removed. Rows with a missing target are not usable for supervised learning, and target values must be binary. Missing numeric predictors are filled with their median; missing categorical predictors use their most frequent value. The identifier is excluded from the modeling dataset.

In [ ]:
df = df_raw.copy()
rows_before = len(df)

if TARGET not in df.columns:
    raise KeyError(f"Required target column {TARGET!r} is missing")
if df[TARGET].isna().any():
    df = df.dropna(subset=[TARGET]).copy()
invalid_targets = set(df[TARGET].unique()) - {0, 1}
if invalid_targets:
    raise ValueError(f"Target must contain only 0 and 1; found {invalid_targets}")

if ID_COLUMN in df.columns:
    df = df.drop_duplicates(subset=ID_COLUMN, keep="first")
df = df.drop_duplicates().copy()

feature_columns = [column for column in df.columns if column not in {TARGET, ID_COLUMN}]
numeric_columns = df[feature_columns].select_dtypes(include="number").columns.tolist()
categorical_columns = [column for column in feature_columns if column not in numeric_columns]

for column in numeric_columns:
    median = df[column].median()
    df[column] = df[column].fillna(median)
for column in categorical_columns:
    mode = df[column].mode(dropna=True)
    if df[column].isna().any() and mode.empty:
        raise ValueError(f"Cannot infer a fill value for empty categorical column {column!r}")
    if not mode.empty:
        df[column] = df[column].fillna(mode.iloc[0])

if ID_COLUMN in df.columns:
    df = df.drop(columns=ID_COLUMN)

if df.isna().any().any():
    remaining = df.columns[df.isna().any()].tolist()
    raise ValueError(f"Cleaning left missing values in: {remaining}")
assert set(df[TARGET].unique()) <= {0, 1}

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Rows retained: {len(df):,} of {rows_before:,}")
print(f"Cleaned dataset saved to: {CLEAN_PATH}")
print(f"Final shape: {df.shape}; missing values: {int(df.isna().sum().sum()):,}")
display(df.head())